# AML Copilot - Test Scorecard Dashboard

Interactive dashboard for viewing unified test results across all test types:
- Conversation Tests (multi-turn behavior)
- Evaluation Tests (AML knowledge)
- System Tests (basic behavior)

**Features:**
- Unified scorecard view
- Drill-down by test type
- Category-level analysis
- Historical trend comparison
- Quality gate status

In [ ]:
import sys
from pathlib import Path
import json
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Scorecard Data

In [ ]:
# Load latest scorecard
results_dir = project_root / "tests" / "results"
scorecard_file = results_dir / "test_scorecard_latest.json"

if not scorecard_file.exists():
    print("❌ No scorecard found!")
    print("   Run: make test-scorecard")
else:
    with open(scorecard_file, 'r') as f:
        scorecard = json.load(f)
    
    print("✅ Scorecard loaded successfully")
    print(f"   Generated: {scorecard['generated_at']}")

## 2. Overall Summary

In [ ]:
# Display overall metrics
overall = scorecard['overall']

print("="*70)
print("OVERALL TEST SUMMARY")
print("="*70)
print(f"\n{overall['status']}")
print(f"\nTotal Tests: {overall['total_tests']}")
print(f"Passed: {overall['total_passed']}")
print(f"Pass Rate: {overall['overall_pass_rate']:.1%}")
print("\n" + "="*70)

In [ ]:
# Create overall pass rate visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pass rate gauge
pass_rate = overall['overall_pass_rate']
colors = ['#ff4444' if pass_rate < 0.5 else '#ffaa44' if pass_rate < 0.8 else '#44ff44']

ax1.barh(['Overall'], [pass_rate], color=colors)
ax1.set_xlim(0, 1)
ax1.set_xlabel('Pass Rate')
ax1.set_title('Overall Pass Rate', fontsize=14, fontweight='bold')
ax1.axvline(x=0.9, color='green', linestyle='--', label='Production Target (90%)')
ax1.legend()

# Format as percentage
for i, v in enumerate([pass_rate]):
    ax1.text(v + 0.02, i, f'{v:.1%}', va='center', fontweight='bold')

# Pass/Fail distribution pie chart
passed = overall['total_passed']
failed = overall['total_tests'] - passed

ax2.pie([passed, failed], labels=['Passed', 'Failed'], autopct='%1.1f%%',
        colors=['#44ff44', '#ff4444'], startangle=90)
ax2.set_title('Test Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. By Test Type Breakdown

In [ ]:
# Create DataFrame for test types
by_type = scorecard['by_test_type']

test_type_data = []
for test_type, metrics in by_type.items():
    if metrics['available']:
        test_type_data.append({
            'Test Type': test_type.replace('_', ' ').title(),
            'Total': metrics['total'],
            'Passed': metrics['passed'],
            'Failed': metrics['failed'],
            'Pass Rate': metrics['pass_rate'],
            'Status': metrics['status']
        })
    else:
        test_type_data.append({
            'Test Type': test_type.replace('_', ' ').title(),
            'Total': 0,
            'Passed': 0,
            'Failed': 0,
            'Pass Rate': 0.0,
            'Status': '⚠️ Not Run'
        })

df_by_type = pd.DataFrame(test_type_data)

# Style the DataFrame
def color_status(val):
    if '✅' in str(val):
        return 'background-color: #d4edda'
    elif '❌' in str(val):
        return 'background-color: #f8d7da'
    elif '⚠️' in str(val):
        return 'background-color: #fff3cd'
    return ''

styled_df = df_by_type.style.applymap(color_status, subset=['Status'])
display(styled_df)

In [ ]:
# Visualize pass rates by test type
fig, ax = plt.subplots(figsize=(10, 6))

# Filter only available test types
available_types = df_by_type[df_by_type['Total'] > 0]

if not available_types.empty:
    colors_map = available_types['Pass Rate'].apply(
        lambda x: '#44ff44' if x >= 0.8 else '#ffaa44' if x >= 0.5 else '#ff4444'
    )
    
    bars = ax.barh(available_types['Test Type'], available_types['Pass Rate'], color=colors_map)
    
    ax.set_xlim(0, 1)
    ax.set_xlabel('Pass Rate', fontsize=12)
    ax.set_title('Pass Rate by Test Type', fontsize=14, fontweight='bold')
    
    # Add target lines
    ax.axvline(x=0.9, color='green', linestyle='--', alpha=0.7, label='Excellent (90%)')
    ax.axvline(x=0.7, color='orange', linestyle='--', alpha=0.7, label='Good (70%)')
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Needs Improvement (50%)')
    
    # Add percentage labels
    for i, (idx, row) in enumerate(available_types.iterrows()):
        ax.text(row['Pass Rate'] + 0.02, i, f"{row['Pass Rate']:.1%}", 
                va='center', fontweight='bold')
    
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No test results available yet. Run tests first.")

## 4. Quality Gates Status

In [ ]:
# Display quality gates
gates = scorecard['quality_gates']

gate_data = [
    {'Gate': 'Production Ready', 'Status': '✅ YES' if gates['production_ready'] else '❌ NO', 'Threshold': '90%'},
    {'Gate': 'Conversation Ready', 'Status': '✅ YES' if gates['conversation_ready'] else '❌ NO', 'Threshold': '80%'},
    {'Gate': 'Evaluation Ready', 'Status': '✅ YES' if gates['evaluation_ready'] else '❌ NO', 'Threshold': '95%'},
    {'Gate': 'System Ready', 'Status': '✅ YES' if gates['system_ready'] else '❌ NO', 'Threshold': '90%'}
]

df_gates = pd.DataFrame(gate_data)

# Style gates
styled_gates = df_gates.style.applymap(color_status, subset=['Status'])
display(styled_gates)

In [ ]:
# Visualize quality gates
fig, ax = plt.subplots(figsize=(10, 5))

gate_names = [g['Gate'] for g in gate_data]
gate_status = [1 if '✅' in g['Status'] else 0 for g in gate_data]
gate_colors = ['#44ff44' if s == 1 else '#ff4444' for s in gate_status]

ax.barh(gate_names, gate_status, color=gate_colors)
ax.set_xlim(0, 1)
ax.set_xlabel('Status')
ax.set_title('Quality Gates Status', fontsize=14, fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['FAIL', 'PASS'])

# Add threshold labels
for i, (name, status, data) in enumerate(zip(gate_names, gate_status, gate_data)):
    label = 'PASS' if status == 1 else 'FAIL'
    ax.text(status + 0.02 if status == 0 else status - 0.15, i, 
            f"{label} (need {data['Threshold']})", va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Drill-Down: Conversation Tests

In [ ]:
# Load conversation test details
conv_file = results_dir / "conversation_tests_latest.json"

if conv_file.exists():
    with open(conv_file, 'r') as f:
        conv_results = json.load(f)
    
    print("\n" + "="*70)
    print("CONVERSATION TESTS - DETAILED BREAKDOWN")
    print("="*70)
    print(f"Total: {conv_results['total']}")
    print(f"Passed: {conv_results['passed']} ({conv_results['pass_rate']:.1%})")
    print(f"Failed: {conv_results['failed']}")
    
    # Category breakdown
    print("\nBy Category:")
    category_data = []
    for cat, stats in conv_results.get('category_stats', {}).items():
        pass_rate = stats['passed'] / stats['total'] if stats['total'] > 0 else 0
        category_data.append({
            'Category': cat.replace('_', ' ').title(),
            'Total': stats['total'],
            'Passed': stats['passed'],
            'Failed': stats['failed'],
            'Pass Rate': pass_rate
        })
        print(f"  {cat}: {stats['passed']}/{stats['total']} ({pass_rate:.1%})")
    
    # Visualize categories
    if category_data:
        df_categories = pd.DataFrame(category_data)
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        colors = df_categories['Pass Rate'].apply(
            lambda x: '#44ff44' if x >= 0.8 else '#ffaa44' if x >= 0.5 else '#ff4444'
        )
        
        bars = ax.barh(df_categories['Category'], df_categories['Pass Rate'], color=colors)
        ax.set_xlim(0, 1)
        ax.set_xlabel('Pass Rate')
        ax.set_title('Conversation Tests - Pass Rate by Category', fontsize=14, fontweight='bold')
        
        for i, (idx, row) in enumerate(df_categories.iterrows()):
            ax.text(row['Pass Rate'] + 0.02, i, 
                    f"{row['Pass Rate']:.1%} ({row['Passed']}/{row['Total']})",
                    va='center', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ No conversation test results found")

## 6. Drill-Down: Evaluation Tests

In [ ]:
# Load evaluation test details
eval_file = results_dir / "evaluation_tests_latest.json"

if eval_file.exists():
    with open(eval_file, 'r') as f:
        eval_results = json.load(f)
    
    print("\n" + "="*70)
    print("EVALUATION TESTS - DETAILED BREAKDOWN")
    print("="*70)
    print(f"Total: {eval_results['total_cases']}")
    print(f"Passed: {eval_results['passed']} ({eval_results['pass_rate']:.1%})")
    print(f"Failed: {eval_results['failed']}")
    print(f"\nAverage Scores:")
    print(f"  Overall: {eval_results['avg_overall_score']:.1f}/100")
    print(f"  Correctness: {eval_results['avg_correctness_score']:.2f}")
    print(f"  Completeness: {eval_results['avg_completeness_score']:.2f}")
    print(f"  Hallucination: {eval_results['avg_hallucination_score']:.2f}")
    
    # Visualize scores
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Score breakdown
    score_data = {
        'Correctness': eval_results['avg_correctness_score'],
        'Completeness': eval_results['avg_completeness_score'],
        'Hallucination': eval_results['avg_hallucination_score']
    }
    
    ax1.bar(score_data.keys(), score_data.values(), color=['#4488ff', '#44ff88', '#ff8844'])
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Score (0-1)')
    ax1.set_title('Evaluation Metrics Breakdown', fontsize=12, fontweight='bold')
    ax1.axhline(y=0.8, color='green', linestyle='--', label='Target (0.8)')
    ax1.legend()
    
    # Overall score gauge
    ax2.barh(['Overall Score'], [eval_results['avg_overall_score']], color='#4488ff')
    ax2.set_xlim(0, 100)
    ax2.set_xlabel('Score (0-100)')
    ax2.set_title('Average Overall Score', fontsize=12, fontweight='bold')
    ax2.text(eval_results['avg_overall_score'] + 2, 0, 
             f"{eval_results['avg_overall_score']:.1f}", va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No evaluation test results found")

## 7. Drill-Down: System Tests

In [ ]:
# Load system test details
sys_file = results_dir / "system_tests_latest.json"

if sys_file.exists():
    with open(sys_file, 'r') as f:
        sys_results = json.load(f)
    
    print("\n" + "="*70)
    print("SYSTEM TESTS - DETAILED BREAKDOWN")
    print("="*70)
    print(f"Total: {sys_results['total']}")
    print(f"Passed: {sys_results['passed']} ({sys_results['pass_rate']:.1%})")
    print(f"Failed: {sys_results['failed']}")
    print(f"Errors: {sys_results['errors']}")
    
    # Category breakdown
    print("\nBy Category:")
    sys_category_data = []
    for cat, stats in sys_results.get('category_stats', {}).items():
        total = stats['total']
        passed = stats['passed']
        pass_rate = passed / total if total > 0 else 0
        sys_category_data.append({
            'Category': cat.replace('_', ' ').title(),
            'Total': total,
            'Passed': passed,
            'Pass Rate': pass_rate
        })
        print(f"  {cat}: {passed}/{total} ({pass_rate:.1%})")
    
    # Visualize
    if sys_category_data:
        df_sys_cat = pd.DataFrame(sys_category_data)
        
        fig, ax = plt.subplots(figsize=(10, 5))
        
        colors = df_sys_cat['Pass Rate'].apply(
            lambda x: '#44ff44' if x >= 0.8 else '#ffaa44' if x >= 0.5 else '#ff4444'
        )
        
        ax.barh(df_sys_cat['Category'], df_sys_cat['Pass Rate'], color=colors)
        ax.set_xlim(0, 1)
        ax.set_xlabel('Pass Rate')
        ax.set_title('System Tests - Pass Rate by Category', fontsize=14, fontweight='bold')
        
        for i, (idx, row) in enumerate(df_sys_cat.iterrows()):
            ax.text(row['Pass Rate'] + 0.02, i,
                    f"{row['Pass Rate']:.1%} ({row['Passed']}/{row['Total']})",
                    va='center', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ No system test results found")

## 8. Recommendations

In [ ]:
# Display recommendations
print("\n" + "="*70)
print("RECOMMENDATIONS")
print("="*70)

for i, rec in enumerate(scorecard['recommendations'], 1):
    print(f"\n{i}. {rec}")

print("\n" + "="*70)

## 9. Export Summary Report

In [ ]:
# Generate HTML report
html_report = f"""
<html>
<head>
    <title>AML Copilot Test Scorecard</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1 {{ color: #333; }}
        .metric {{ padding: 15px; margin: 10px 0; border-radius: 5px; }}
        .pass {{ background-color: #d4edda; }}
        .warning {{ background-color: #fff3cd; }}
        .fail {{ background-color: #f8d7da; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
        th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
        th {{ background-color: #4CAF50; color: white; }}
    </style>
</head>
<body>
    <h1>AML Copilot - Test Scorecard</h1>
    <p><strong>Generated:</strong> {scorecard['generated_at']}</p>
    
    <div class="metric {'pass' if overall['overall_pass_rate'] >= 0.8 else 'warning' if overall['overall_pass_rate'] >= 0.5 else 'fail'}">
        <h2>{overall['status']}</h2>
        <p><strong>Overall Pass Rate:</strong> {overall['overall_pass_rate']:.1%} ({overall['total_passed']}/{overall['total_tests']} tests)</p>
    </div>
    
    <h2>By Test Type</h2>
    {df_by_type.to_html(index=False)}
    
    <h2>Quality Gates</h2>
    {df_gates.to_html(index=False)}
    
    <h2>Recommendations</h2>
    <ol>
        {''.join([f'<li>{rec}</li>' for rec in scorecard['recommendations']])}
    </ol>
</body>
</html>
"""

# Save HTML report
html_file = results_dir / "test_scorecard_report.html"
with open(html_file, 'w') as f:
    f.write(html_report)

print(f"\n📊 HTML report saved to: {html_file}")
print(f"   Open in browser to view")

## Next Steps

**To improve test scores:**
1. Run all test types: `make test`
2. Focus on failed categories identified above
3. Re-run scorecard: `make test-scorecard`
4. Refresh this notebook to see updated results

**For detailed investigation:**
- View individual test results in `tests/results/*_latest.json`
- Use drill-down sections above to identify specific failures
- Check recommendations for prioritized improvements